# Approximate Unlearning for Any Book

Generalised implementation of [Eldan & Russinovich (2023)](https://arxiv.org/abs/2310.02238).

**Goal**: surgically remove a target book's knowledge from an LLM while preserving general capabilities — without any book-specific hardcoded knowledge.

**Pipeline**:
1. Extract and chunk text from any PDF(s)
2. Train a *reinforced* LoRA adapter — amplifies the book's knowledge
3. Detect *anchor tokens* — positions where the model diverges from baseline (logit-ratio scan)
4. Fine-tune an *unlearn* adapter on soft replacement labels (paper formula)
5. Evaluate with fully unsupervised metrics: anchor recall drop, perplexity delta, general benchmarks

**All evaluation is unsupervised** — the book's own content drives the metrics.

---
**Runtime**: GPU required.

**Before running**: upload your book PDF(s) to the path set in `PDF_GLOB` below.

## 0 · Setup

In [ ]:
!pip install -q \
    pymupdf \
    transformers \
    datasets \
    peft \
    bitsandbytes \
    accelerate \
    trl \
    lm-eval \
    huggingface_hub \
    "torchao>=0.16.0"

In [ ]:
import ast, json, math, re
from pathlib import Path
from typing import List

import fitz                          # pymupdf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset, load_from_disk
from IPython.display import HTML, display
from peft import (
    LoraConfig, PeftModel, TaskType,
    get_peft_model, prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments,
)

assert torch.cuda.is_available(), "Go to Runtime → Change runtime type → GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from huggingface_hub import login

# Option A (recommended): Colab Secrets — left sidebar → key icon → add secret named HF_TOKEN
# Option B: paste your token directly below (get one at https://huggingface.co/settings/tokens)
HF_TOKEN = ""  # ← paste here only if not using Colab Secrets

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN or not HF_TOKEN.strip():
    raise ValueError(
        "HF_TOKEN is empty.\n"
        "Fix one of two ways:\n"
        "  A) Left sidebar → key icon → add secret named HF_TOKEN\n"
        "  B) Paste your token into HF_TOKEN = \"hf_...\" above"
    )

login(token=HF_TOKEN.strip())
print("Logged in to HuggingFace.")

In [ ]:
# Mount Drive so trained artifacts survive session restarts.
# If Drive is unavailable, everything is saved locally at /content/unlearning/.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/book_unlearning")
    print(f"Using Drive: {BASE_DIR}")
except Exception:
    BASE_DIR = Path("/content/book_unlearning")
    print(f"No Drive — saving locally: {BASE_DIR}")

BASE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Configuration

MODEL_NAME = "meta-llama/Meta-Llama-3-8B"
PDF_GLOB   = "/content/*.pdf"

CHUNK_SIZE   = 512
EVAL_HOLDOUT = 0.1

# Reinforced adapter
REINF_LR         = 2e-4
REINF_EPOCHS     = 1
REINF_BATCH      = 8
REINF_GRAD_ACCUM = 2    # effective batch = 8×2 = 16 (matches paper)

# Anchor detection — number of chunks to process per forward pass
ANCHOR_BATCH     = 8

# Anchor threshold
ANCHOR_THRESHOLD = 2.0

# Unlearning
ALPHA              = 5.0
UNLEARN_LR         = 2e-5
UNLEARN_EPOCHS     = 3
UNLEARN_GRAD_ACCUM = 4

# Artifact paths
CORPUS_DIR     = BASE_DIR / "corpus"
REINFORCED_DIR = BASE_DIR / "reinforced_adapter"
ANCHORED_DIR   = BASE_DIR / "anchored"
UNLEARNED_DIR  = BASE_DIR / "unlearned_adapter"

print("Config ready.")

In [ ]:
# Results persistence — stores numbers from both eval runs for comparison plots
RESULTS_FILE  = BASE_DIR / "eval_results.json"
FIGURES_DIR   = BASE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def load_results() -> dict:
    if RESULTS_FILE.exists():
        with open(RESULTS_FILE) as f:
            return json.load(f)
    return {}

def save_result(run_label: str, data: dict) -> None:
    results = load_results()
    results[run_label] = data
    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved '{run_label}'. Stored runs: {list(results.keys())}")

all_results = load_results()
print(f"Existing result runs: {list(all_results.keys()) or 'none yet'}")

---
## Step 1 · Load & Chunk Corpus

Works for any PDF. Extracts text, strips formatting noise, tokenizes the full text, then slices into 512-token non-overlapping windows — matching the paper's context length.

The last `EVAL_HOLDOUT` fraction of chunks is reserved for evaluation and never used during training.

In [ ]:
pdfs = sorted(Path("/content").glob("*.pdf"))
if not pdfs:
    from google.colab import files
    print("No PDFs found — uploading now...")
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(f"/content/{name}").write_bytes(data)
    pdfs = sorted(Path("/content").glob("*.pdf"))

print(f"PDFs to process: {[p.name for p in pdfs]}")

In [ ]:
def extract_pdf_text(pdf_path: Path) -> str:
    pages = []
    with fitz.open(str(pdf_path)) as doc:
        for page in doc:
            text = page.get_text().strip()
            if text:
                pages.append(text)
    return "\n".join(pages)


def clean_text(raw: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', raw)
    text = re.sub(r'(?m)^\s*\d+\s*$', '', text)  # bare page numbers
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


def chunk_by_tokens(text: str, tokenizer) -> List[str]:
    """Tokenize full text, then slice into non-overlapping CHUNK_SIZE windows."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for start in range(0, len(ids), CHUNK_SIZE):
        window = ids[start : start + CHUNK_SIZE]
        if len(window) < 32:   # drop tiny trailing fragments
            continue
        chunks.append(tokenizer.decode(window))
    return chunks


if CORPUS_DIR.exists():
    print(f"Corpus already exists — loading from {CORPUS_DIR}")
    full_corpus = load_from_disk(str(CORPUS_DIR))
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    all_chunks = []
    for pdf in pdfs:
        print(f"Extracting: {pdf.name}")
        chunks = chunk_by_tokens(clean_text(extract_pdf_text(pdf)), tok)
        print(f"  → {len(chunks)} chunks")
        all_chunks.extend(chunks)

    full_corpus = Dataset.from_dict({"text": all_chunks})
    full_corpus.save_to_disk(str(CORPUS_DIR))
    del tok

# Train / eval split
split = full_corpus.train_test_split(test_size=EVAL_HOLDOUT, seed=42)
train_corpus = split["train"]
eval_corpus  = split["test"]

print(f"\nCorpus: {len(full_corpus)} chunks total")
print(f"  Training : {len(train_corpus)} chunks")
print(f"  Eval     : {len(eval_corpus)} chunks (held out)")
print(f"\nSample:\n{train_corpus[0]['text'][:400]}")

---
## Step 2 · Train Reinforced Adapter

Fine-tunes a LoRA adapter on the book corpus. This makes the model over-confident about book-specific tokens. The divergence between this adapter and the frozen base is the signal used in Steps 3 and 4 — no book-specific knowledge is hardcoded anywhere.



In [ ]:
reinf_loss_history = []   # populated below; persists for the loss-curve plot

if REINFORCED_DIR.exists():
    print(f"Reinforced adapter found at {REINFORCED_DIR} — skipping training.")
    loss_log_path = BASE_DIR / "reinf_loss_log.json"
    if loss_log_path.exists():
        with open(loss_log_path) as f:
            reinf_loss_history = json.load(f)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_for_clm(batch):
        enc = tokenizer(batch["text"], truncation=True,
                        max_length=CHUNK_SIZE, padding="max_length")
        enc["labels"] = enc["input_ids"].copy()
        return enc

    tokenized = train_corpus.map(
        tokenize_for_clm, batched=True, remove_columns=train_corpus.column_names
    )
    tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map="auto",
    )
    base.config.use_cache = False

    model = get_peft_model(base, LoraConfig(
        r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.0, bias="none", task_type=TaskType.CAUSAL_LM,
    ))
    model.print_trainable_parameters()

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="/content/reinf_ckpts",
            num_train_epochs=REINF_EPOCHS,
            per_device_train_batch_size=REINF_BATCH,
            gradient_accumulation_steps=REINF_GRAD_ACCUM,
            gradient_checkpointing=True,
            learning_rate=REINF_LR,
            bf16=True,
            logging_steps=10,
            save_strategy="no",
            report_to="none",
            remove_unused_columns=False,
            dataloader_pin_memory=False,
            optim="adamw_torch_fused",
        ),
        train_dataset=tokenized,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    trainer.train()

    # Save loss log for the plot cell
    reinf_loss_history = [
        {"step": e["step"], "loss": e["loss"]}
        for e in trainer.state.log_history if "loss" in e
    ]
    with open(BASE_DIR / "reinf_loss_log.json", "w") as f:
        json.dump(reinf_loss_history, f)

    model.save_pretrained(str(REINFORCED_DIR))
    tokenizer.save_pretrained(str(REINFORCED_DIR))
    print(f"Reinforced adapter saved to: {REINFORCED_DIR}")

    del model, base, trainer, tokenized
    torch.cuda.empty_cache()

In [ ]:
# Figure 3a — Reinforced Training Loss Curve
if reinf_loss_history:
    steps  = [e["step"] for e in reinf_loss_history]
    losses = [e["loss"] for e in reinf_loss_history]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, losses, color="#4C72B0", linewidth=1.5, label="Train loss")

    # Smoothed trend
    window = max(1, len(losses) // 10)
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    ax.plot(steps[window-1:], smoothed, color="#C44E52", linewidth=2,
            linestyle="--", label=f"Smoothed (w={window})")

    ax.set_xlabel("Optimiser Step")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.set_title("Figure 3a — Reinforced Adapter Training Loss")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(FIGURES_DIR / "fig3a_reinf_loss.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIGURES_DIR / 'fig3a_reinf_loss.png'}")
else:
    print("No loss history available — run training first.")

---
## Step 3 · Detect Anchor Tokens

For every token position in the training corpus, compute:

$$\text{ratio}_i = \frac{p_{\text{reinforced}}(x_i)}{p_{\text{baseline}}(x_i)}$$

Positions where `ratio ≥ ANCHOR_THRESHOLD` are *anchor tokens* — the model learned these specifically from the book. No named entity list or book-specific knowledge is needed: the model itself reveals what it over-knows.



In [ ]:
if ANCHORED_DIR.exists():
    print(f"Anchored dataset found at {ANCHORED_DIR} — loading.")
    anchored = load_from_disk(str(ANCHORED_DIR))
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    detect_model = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            dtype=torch.bfloat16,
            device_map="auto",
        ),
        str(REINFORCED_DIR), adapter_name="reinforced", is_trainable=False,
    )
    detect_model.eval()
    device = next(detect_model.parameters()).device

    records = []
    total = len(train_corpus)

    for batch_start in range(0, total, ANCHOR_BATCH):
        batch = train_corpus[batch_start : batch_start + ANCHOR_BATCH]
        texts = batch["text"] if isinstance(batch["text"], list) else [batch["text"]]

        enc = tokenizer(
            texts, return_tensors="pt",
            truncation=True, max_length=CHUNK_SIZE,
            padding=True,
        )
        ids  = enc["input_ids"].to(device)
        attn = enc["attention_mask"].to(device)

        with torch.no_grad():
            with detect_model.disable_adapter():
                p_base = torch.softmax(
                    detect_model(input_ids=ids, attention_mask=attn).logits, dim=-1
                )
            detect_model.set_adapter("reinforced")
            p_reinf = torch.softmax(
                detect_model(input_ids=ids, attention_mask=attn).logits, dim=-1
            )

        for b in range(len(texts)):
            real_len = attn[b].sum().item()
            true_next = ids[b, 1:real_len]
            pos = torch.arange(real_len - 1, device=device)

            p_b = p_base[b, pos, true_next]
            p_r = p_reinf[b, pos, true_next]
            ratio = p_r / (p_b + 1e-9)

            anchor_mask = (ratio >= ANCHOR_THRESHOLD).cpu().tolist()
            anchor_rate = sum(anchor_mask) / max(len(anchor_mask), 1)
            records.append({
                "text": texts[b],
                "anchor_mask": str(anchor_mask),
                "anchor_rate": anchor_rate,
            })

        done = min(batch_start + ANCHOR_BATCH, total)
        if done % 200 < ANCHOR_BATCH or done == total:
            mean_rate = sum(r["anchor_rate"] for r in records) / len(records)
            print(f"[{done}/{total}] mean anchor rate: {mean_rate:.3f}")

        del ids, attn, p_base, p_reinf
        torch.cuda.empty_cache()

    anchored = Dataset.from_dict({
        "text":        [r["text"]        for r in records],
        "anchor_mask": [r["anchor_mask"] for r in records],
        "anchor_rate": [r["anchor_rate"] for r in records],
    })
    anchored.save_to_disk(str(ANCHORED_DIR))

    overall = sum(r["anchor_rate"] for r in records) / len(records)
    print(f"\nAnchor detection complete. Mean anchor rate: {overall:.3f}")

    del detect_model
    torch.cuda.empty_cache()

print(f"Anchored dataset: {len(anchored)} chunks")
print(f"Anchor rates — min: {min(anchored['anchor_rate']):.3f}  "
      f"mean: {sum(anchored['anchor_rate'])/len(anchored):.3f}  "
      f"max: {max(anchored['anchor_rate']):.3f}")

In [ ]:
# Figure 2 — Anchor Token Rate Distribution
rates = anchored["anchor_rate"]
mean_rate = sum(rates) / len(rates)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: histogram
axes[0].hist(rates, bins=40, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].axvline(mean_rate, color="#C44E52", linestyle="--", linewidth=2,
                label=f"Mean: {mean_rate:.3f}")
axes[0].set_xlabel("Anchor Rate (fraction of tokens flagged per chunk)")
axes[0].set_ylabel("Number of Chunks")
axes[0].set_title("Figure 2a — Anchor Rate Distribution")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: anchor rate by chunk position (reveals whether certain chapters are denser)
axes[1].scatter(range(len(rates)), rates, alpha=0.35, s=8, color="#4C72B0")
axes[1].axhline(ANCHOR_THRESHOLD / 10, color="#C44E52", linestyle="--", alpha=0.6,
                label=f"Threshold reference")
axes[1].axhline(mean_rate, color="orange", linestyle="--", alpha=0.8,
                label=f"Mean: {mean_rate:.3f}")
axes[1].set_xlabel("Chunk Index (book order →)")
axes[1].set_ylabel("Anchor Rate")
axes[1].set_title("Figure 2b — Anchor Rate by Narrative Position")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.suptitle(f"Anchor Token Analysis  |  {len(anchored)} training chunks  |  "
             f"threshold={ANCHOR_THRESHOLD}", fontsize=11)
fig.tight_layout()
fig.savefig(str(FIGURES_DIR / "fig2_anchor_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIGURES_DIR / 'fig2_anchor_distribution.png'}")

In [ ]:
# Figure 4 — Anchor Token Highlighting (qualitative)
# Picks the chunk with the highest anchor rate and renders it with anchor tokens
# highlighted in red. Inline HTML in the notebook; PNG saved to Drive.

_tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

top_idx = int(np.argmax(anchored["anchor_rate"]))
sample   = anchored[top_idx]
mask     = ast.literal_eval(sample["anchor_mask"])
token_ids = _tok.encode(sample["text"], add_special_tokens=False)

# Align mask length with token list
n = min(len(mask), len(token_ids) - 1, 200)   # cap at 200 tokens for readability
tokens = [_tok.decode([tid]) for tid in token_ids[:n+1]]

# HTML display (interactive notebook)
html = ["<div style='font-family:monospace;line-height:2;font-size:13px;max-width:950px'>"]
html.append(f"<p><b>Figure 4 — Chunk {top_idx} &nbsp;|&nbsp; "
            f"Anchor rate: {sample['anchor_rate']:.3f} &nbsp;|&nbsp; "
            f"Red = anchor token (logit ratio ≥ {ANCHOR_THRESHOLD})</b></p><p>")
for i, tok in enumerate(tokens[:-1]):
    esc = tok.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    if i < len(mask) and mask[i]:
        html.append(f"<span style='background:#e74c3c;color:#fff;"
                    f"border-radius:3px;padding:1px 4px;margin:1px'>{esc}</span>")
    else:
        html.append(esc)
html.append("</p></div>")
display(HTML("".join(html)))

# PNG version (for report)
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")

n_anchors = sum(mask[:n])
pct = 100 * n_anchors / max(n, 1)
title = (f"Figure 4 — Anchor Token Highlighting\n"
         f"Chunk {top_idx}  |  anchor rate: {sample['anchor_rate']:.3f}  "
         f"|  {n_anchors}/{n} tokens flagged ({pct:.1f}%)")
ax.set_title(title, fontsize=11, pad=10)

# Word-wrap the token string, colour coded via two text blocks
plain_text = "".join(tokens[:n+1])[:600] + " ..."
ax.text(0.01, 0.85, plain_text, transform=ax.transAxes,
        fontsize=8.5, verticalalignment="top", wrap=True,
        family="monospace",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", alpha=0.9))

anchor_patch = mpatches.Patch(color="#e74c3c", label="Anchor token")
base_patch   = mpatches.Patch(color="#ecf0f1", label="Non-anchor token")
ax.legend(handles=[anchor_patch, base_patch], loc="lower right")

fig.tight_layout()
fig.savefig(str(FIGURES_DIR / "fig4_token_highlight.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIGURES_DIR / 'fig4_token_highlight.png'}")
del _tok

---
## Step 4 · Unlearn (Soft Labels)

The replacement target at every position is a **soft label** — the full modified logit distribution:

$$v_{\text{generic}} = v_{\text{baseline}} - \alpha \cdot \text{ReLU}\bigl(v_{\text{reinforced}} - v_{\text{baseline}}\bigr)$$

$$\mathcal{L} = \text{KL}\!\left(\log\sigma(v_{\text{unlearn}})\;\|\;\sigma(v_{\text{generic}})\right)$$

`ReLU` makes suppression one-directional: only tokens the reinforced model boosted get pushed down. Tokens where both models agree are unchanged — preserving general language capability.



In [ ]:
unlearn_loss_history = []   # populated below; persists for the loss-curve plot

if UNLEARNED_DIR.exists():
    print(f"Unlearned adapter found at {UNLEARNED_DIR} — skipping.")
    loss_log_path = BASE_DIR / "unlearn_loss_log.json"
    if loss_log_path.exists():
        with open(loss_log_path) as f:
            unlearn_loss_history = json.load(f)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map="auto",
    )
    base.config.use_cache = False
    base.gradient_checkpointing_enable()

    model = PeftModel.from_pretrained(
        base, str(REINFORCED_DIR), adapter_name="reinforced", is_trainable=False,
    )
    model.add_adapter("unlearn", LoraConfig(
        r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.0, bias="none", task_type=TaskType.CAUSAL_LM,
    ))

    for name, param in model.named_parameters():
        param.requires_grad = "lora_" in name and ".unlearn." in name

    model.set_adapter("unlearn")
    model.train()
    device = next(model.parameters()).device

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=UNLEARN_LR, fused=True,
    )
    optimizer.zero_grad(set_to_none=True)

    total = len(anchored)
    global_step = 0
    print(f"Unlearning: {UNLEARN_EPOCHS} epochs × {total} chunks | α={ALPHA} | lr={UNLEARN_LR}")

    for epoch in range(UNLEARN_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{UNLEARN_EPOCHS}")
        for idx in range(total):
            enc = tokenizer(
                anchored[idx]["text"], return_tensors="pt",
                truncation=True, max_length=CHUNK_SIZE, padding=False,
            )
            ids  = enc["input_ids"].to(device)
            attn = enc["attention_mask"].to(device)

            with torch.no_grad():
                with model.disable_adapter():
                    v_base  = model(input_ids=ids, attention_mask=attn).logits[0]
                model.set_adapter("reinforced")
                v_reinf = model(input_ids=ids, attention_mask=attn).logits[0]

            v_generic = v_base - ALPHA * torch.relu(v_reinf - v_base)
            p_target  = torch.softmax(v_generic, dim=-1).detach()

            model.set_adapter("unlearn")
            logits = model(input_ids=ids, attention_mask=attn).logits[0]

            loss = F.kl_div(
                F.log_softmax(logits, dim=-1), p_target, reduction="batchmean",
            )
            (loss / UNLEARN_GRAD_ACCUM).backward()

            global_step += 1
            unlearn_loss_history.append({"step": global_step, "loss": loss.item()})

            if (idx + 1) % UNLEARN_GRAD_ACCUM == 0 or (idx + 1) == total:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()

            if global_step % 200 == 0 or global_step == total * UNLEARN_EPOCHS:
                print(f"  step {global_step}/{total * UNLEARN_EPOCHS}  loss={loss.item():.5f}")

            del ids, attn, v_base, v_reinf, v_generic, p_target, logits, loss

    with open(BASE_DIR / "unlearn_loss_log.json", "w") as f:
        json.dump(unlearn_loss_history, f)

    model.set_adapter("unlearn")
    model.save_pretrained(str(UNLEARNED_DIR), selected_adapters=["unlearn"])
    tokenizer.save_pretrained(str(UNLEARNED_DIR))
    print(f"\nUnlearned adapter saved to: {UNLEARNED_DIR}")

    del model, base, optimizer
    torch.cuda.empty_cache()

In [ ]:
# Figure 3b — Unlearning KL Loss Curve
if unlearn_loss_history:
    steps  = [e["step"] for e in unlearn_loss_history]
    losses = [e["loss"] for e in unlearn_loss_history]
    n_per_epoch = len(steps) // UNLEARN_EPOCHS

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(steps, losses, color="#4C72B0", linewidth=0.8, alpha=0.5, label="KL loss (per step)")

    window = max(1, len(losses) // 20)
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    ax.plot(steps[window-1:], smoothed, color="#C44E52", linewidth=2,
            label=f"Smoothed (w={window})")

    for e in range(1, UNLEARN_EPOCHS):
        ax.axvline(e * n_per_epoch, color="gray", linestyle=":", alpha=0.6)
        ax.text(e * n_per_epoch + 2, max(losses)*0.95, f"Epoch {e+1}",
                fontsize=8, color="gray")

    ax.set_xlabel("Step")
    ax.set_ylabel("KL Divergence Loss")
    ax.set_title(f"Figure 3b — Unlearning Loss  (α={ALPHA}, lr={UNLEARN_LR})")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(FIGURES_DIR / "fig3b_unlearn_loss.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIGURES_DIR / 'fig3b_unlearn_loss.png'}")
else:
    print("No loss history — run unlearning first.")

---
## Step 5 · Evaluate (Unsupervised)

All metrics are derived from the book itself — no hardcoded questions or expected answers.

| Metric | What it measures | After unlearning |
|---|---|---|
| **Anchor recall** | % of anchor tokens the model still predicts correctly | Should drop |
| **Book perplexity** | How surprised the model is by held-out book text | Should rise |
| **Control perplexity** | How surprised the model is by non-book text | Should stay flat |
| **Forgetting index** | Normalised anchor recall drop vs. baseline | Higher = more forgotten |
| **General benchmarks** | arc_easy, boolq, winogrande accuracy | Should stay flat |

Run the evaluation block **twice** — once before training (base model) and once after (unlearned adapter) — to compare.

In [ ]:
# Evaluation target
# None   → base model (run this first to get baseline numbers)
# str(UNLEARNED_DIR / "unlearn")  → unlearned model

EVAL_ADAPTER = None
EVAL_ADAPTER = str(UNLEARNED_DIR / "unlearn")   # ← uncomment after training

CONTROL_TEXT = (
    "The mitochondria is the powerhouse of the cell and produces ATP through "
    "oxidative phosphorylation. Water boils at one hundred degrees Celsius at "
    "standard atmospheric pressure. The French Revolution began in 1789 and "
    "fundamentally transformed European political structures."
)

print(f"Evaluating: {'base model' if EVAL_ADAPTER is None else EVAL_ADAPTER}")

In [ ]:
eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)
eval_model = (
    PeftModel.from_pretrained(eval_base, EVAL_ADAPTER, is_trainable=False)
    if EVAL_ADAPTER else eval_base
)
eval_tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if eval_tok.pad_token is None:
    eval_tok.pad_token = eval_tok.eos_token

eval_model.eval()
eval_device = next(eval_model.parameters()).device
print("Eval model loaded.")

In [ ]:
# Metric 1: Anchor token recall
# For each chunk in the held-out eval set, compute what fraction of anchor-flagged
# positions the model still predicts correctly. Drop = successful forgetting.

# Re-detect anchor masks for the eval set using the saved reinforced adapter
eval_reinforced = PeftModel.from_pretrained(
    eval_base, str(REINFORCED_DIR), adapter_name="reinforced", is_trainable=False,
)
eval_reinforced.eval()

correct_at_anchors = []
total_anchors = 0

for row in eval_corpus.select(range(min(50, len(eval_corpus)))):
    enc = eval_tok(
        row["text"], return_tensors="pt",
        truncation=True, max_length=CHUNK_SIZE, padding=False,
    )
    ids  = enc["input_ids"].to(eval_device)
    attn = enc["attention_mask"].to(eval_device)
    true_next = ids[0, 1:]
    seq = ids.shape[1]

    with torch.no_grad():
        # Detect anchors from eval set using reinforced adapter
        with eval_reinforced.disable_adapter():
            p_base  = torch.softmax(eval_reinforced(input_ids=ids, attention_mask=attn).logits[0], dim=-1)
        eval_reinforced.set_adapter("reinforced")
        p_reinf = torch.softmax(eval_reinforced(input_ids=ids, attention_mask=attn).logits[0], dim=-1)

        # Restore eval_model to the unlearn adapter before making predictions
        if EVAL_ADAPTER is not None and hasattr(eval_model, "set_adapter"):
            eval_model.set_adapter("default")


        pos   = torch.arange(seq - 1, device=eval_device)
        ratio = p_reinf[pos, true_next] / (p_base[pos, true_next] + 1e-9)
        is_anchor = ratio >= ANCHOR_THRESHOLD

        # Predict using the eval model (base or unlearned)
        eval_logits = eval_model(input_ids=ids, attention_mask=attn).logits[0]
        predictions = eval_logits[:-1].argmax(dim=-1)   # predictions for positions 0..seq-2

    anchor_positions = is_anchor.nonzero(as_tuple=True)[0]
    if len(anchor_positions) > 0:
        hits = (predictions[anchor_positions] == true_next[anchor_positions]).sum().item()
        correct_at_anchors.append(hits / len(anchor_positions))
        total_anchors += len(anchor_positions)

    torch.cuda.empty_cache()

del eval_reinforced
torch.cuda.empty_cache()

anchor_recall = sum(correct_at_anchors) / max(len(correct_at_anchors), 1)
print(f"Anchor token recall : {anchor_recall:.4f}  ({anchor_recall*100:.1f}% of anchor positions predicted correctly)")
print(f"Total anchor tokens : {total_anchors}")

In [ ]:
# Metric 2: Perplexity

# Restore eval_model to the unlearn adapter before perplexity computation
if EVAL_ADAPTER is not None and hasattr(eval_model, "set_adapter"):
    eval_model.set_adapter("default")

def compute_perplexity(text: str) -> float:
    enc = eval_tok(
        text, return_tensors="pt", truncation=True, max_length=CHUNK_SIZE,
    ).to(eval_device)
    with torch.no_grad():
        loss = eval_model(input_ids=enc["input_ids"], labels=enc["input_ids"]).loss
    return math.exp(loss.item())


# Average perplexity over several held-out book chunks
book_ppls = [compute_perplexity(eval_corpus[i]["text"])
             for i in range(min(10, len(eval_corpus)))]
book_ppl  = sum(book_ppls) / len(book_ppls)
ctrl_ppl  = compute_perplexity(CONTROL_TEXT)

print("Perplexity")
print("-" * 45)
print(f"Book text (held-out): {book_ppl:8.2f}  ↑ after unlearning = good")
print(f"Control text:         {ctrl_ppl:8.2f}  should stay flat")


In [ ]:
label = "base_model" if EVAL_ADAPTER is None else "unlearned_model"
display_label = "Base Model" if EVAL_ADAPTER is None else "Unlearned Model"

print(f"\n{'='*50}")
print(f"  Results: {display_label}")
print(f"{'='*50}")
print(f"  Anchor recall      : {anchor_recall:.4f}")
print(f"  Book perplexity    : {book_ppl:.2f}")
print(f"  Control perplexity : {ctrl_ppl:.2f}")
print(f"{'='*50}")

# Persist so the comparison plot can read both runs
save_result(label, {
    "anchor_recall": anchor_recall,
    "book_ppl":      book_ppl,
    "ctrl_ppl":      ctrl_ppl,
})

In [ ]:
del eval_model, eval_base
torch.cuda.empty_cache()

from lm_eval import evaluator

model_args = {"pretrained": MODEL_NAME, "dtype": "bfloat16"}
if EVAL_ADAPTER:
    model_args["peft"] = EVAL_ADAPTER

results = evaluator.simple_evaluate(
    model="hf", model_args=model_args,
    tasks=["arc_easy", "boolq", "winogrande"],
    num_fewshot=0, batch_size=8, device="cuda:0",
    limit=100, log_samples=False, bootstrap_iters=0,
)

bench_scores = {}
print(f"\nGeneral Benchmarks — {display_label}")
print(f"{'Task':<15} {'Accuracy':>10}")
print("-" * 28)
for task in ["arc_easy", "boolq", "winogrande"]:
    m   = results["results"].get(task, {})
    acc = m.get("acc,none", m.get("acc", float("nan")))
    bench_scores[task] = acc
    print(f"{task:<15} {acc:>10.4f}")

# Merge benchmark scores into the saved result for this run
run_data = load_results().get(label, {})
run_data.update(bench_scores)
save_result(label, run_data)

---
## Figures 1 & 5 — Before / After Comparison

Run the eval section **twice** (once with `EVAL_ADAPTER = None`, once with the unlearned adapter path) before running this cell. Both runs must be saved to `eval_results.json` on Drive.

In [ ]:
all_results = load_results()

if "base_model" not in all_results or "unlearned_model" not in all_results:
    print("Need both 'base_model' and 'unlearned_model' runs saved.")
    print(f"Currently have: {list(all_results.keys())}")
else:
    base = all_results["base_model"]
    unlearned = all_results["unlearned_model"]

    COLORS = {"Base Model": "#4C72B0", "Unlearned": "#C44E52"}

    # ── Figure 1 — Core Unlearning Metrics (anchor recall + perplexity) ────────
    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    fig.suptitle("Figure 1 — Unlearning Results: Base Model vs. Unlearned Model",
                 fontsize=13, fontweight="bold")

    metrics = [
        ("anchor_recall", "Anchor Token Recall ↓", "Lower = more forgotten"),
        ("book_ppl",      "Book Perplexity ↑",      "Higher = more forgotten"),
        ("ctrl_ppl",      "Control Perplexity →",   "Should stay flat"),
    ]
    for ax, (key, title, subtitle) in zip(axes, metrics):
        vals   = [base.get(key, 0), unlearned.get(key, 0)]
        colors = [COLORS["Base Model"], COLORS["Unlearned"]]
        bars   = ax.bar(["Base Model", "Unlearned"], vals, color=colors,
                        width=0.5, edgecolor="white")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
        ax.set_title(f"{title}\n{subtitle}", fontsize=10)
        ax.set_ylim(0, max(vals) * 1.18)
        ax.grid(axis="y", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    fig.savefig(str(FIGURES_DIR / "fig1_unlearning_results.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIGURES_DIR / 'fig1_unlearning_results.png'}")

    # Figure 5 — General Benchmark Retention
    bench_tasks = ["arc_easy", "boolq", "winogrande"]
    base_scores = [base.get(t, 0) for t in bench_tasks]
    unl_scores  = [unlearned.get(t, 0) for t in bench_tasks]

    x = np.arange(len(bench_tasks))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    bars1 = ax.bar(x - width/2, base_scores, width, label="Base Model",
                   color=COLORS["Base Model"], edgecolor="white")
    bars2 = ax.bar(x + width/2, unl_scores,  width, label="Unlearned",
                   color=COLORS["Unlearned"],  edgecolor="white")

    for bars in [bars1, bars2]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(["ARC-Easy", "BoolQ", "Winogrande"], fontsize=11)
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.1)
    ax.set_title("Figure 5 — General Capability Benchmarks (should be preserved)",
                 fontsize=12)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

    # Annotate % change per task
    for i, (b, u) in enumerate(zip(base_scores, unl_scores)):
        delta = (u - b) / max(b, 1e-9) * 100
        color = "#27ae60" if abs(delta) < 2 else "#e74c3c"
        ax.text(i, max(b, u) + 0.06, f"{delta:+.1f}%", ha="center",
                fontsize=9, color=color, fontweight="bold")

    fig.tight_layout()
    fig.savefig(str(FIGURES_DIR / "fig5_benchmark_retention.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIGURES_DIR / 'fig5_benchmark_retention.png'}")

    # Summary table
    print(f"\n{'Metric':<22} {'Base':>10} {'Unlearned':>12} {'Delta':>10}")
    print("-" * 58)
    all_keys = ["anchor_recall", "book_ppl", "ctrl_ppl"] + bench_tasks
    for k in all_keys:
        b_val = base.get(k, float("nan"))
        u_val = unlearned.get(k, float("nan"))
        delta = u_val - b_val
        print(f"{k:<22} {b_val:>10.4f} {u_val:>12.4f} {delta:>+10.4f}")